In [1]:
import importlib
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

warnings.simplefilter('ignore')
project_root = Path.cwd().resolve().parents[1]
sys.path.append(str(project_root / 'src'))

from func_data_build import build_dataset
from func_gibbs.gibbs_notebook_utils import display_hmc_results, display_hmc_posterior_prior, save_idata_map
from func_gibbs.gibbs_wrappers import draws_to_idata
import func_gibbs.gibbs_wrappers as gibbs_module
from func_gibbs.gibbs_wrappers import run_ces

data_dir = project_root / "data"
idat_dir = project_root / "results" / "idata"
tex_root_dir = project_root / "results" / "tex" / "table"
base_fig_dir = project_root / "results" / "fig"

data = build_dataset(data_dir)
data = data.loc['1982-01-01':'2012-12-31'].copy()
data['DATE'] = pd.to_datetime(data.index)

inflation_specs = {
    'ppi': {
        'pi': 'pi_ppi',
        'pi_prev': 'pi_ppi_prev',
        'pi_expect': 'Epi_spf_gdp',
    },
    'cpi': {
        'pi': 'pi_cpi',
        'pi_prev': 'pi_cpi_prev',
        'pi_expect': 'Epi_spf_cpi',
    },
}

x_datasets = {
    'unemp_gap': (
        np.asarray(data['unemp_gap'], dtype=float),
        np.asarray(data['unemp_gap_prev'], dtype=float),
    ),
    'output_gap_BN': (
        np.asarray(data['output_gap_BN'], dtype=float),
        np.asarray(data['output_gap_BN_prev'], dtype=float),
    ),
    'markup_BN_inv': (
        np.asarray(data['markup_BN_inv'], dtype=float),
        np.asarray(data['markup_BN_inv_prev'], dtype=float),
    ),
    'markup_inv': (
        np.asarray(data['markup_inv'], dtype=float),
        np.asarray(data['markup_inv_prev'], dtype=float),
    ),
}
seed = 0
n_iter = 12000
burn = 4000
thin = 5

prior_specs = {
    'alpha': (0.5, 0.2),
    'kappa': (0.1, 0.2),
    'phi_1': (0.7, 0.2),
    'rho': (0.0, 0.5),
}
tex_dir = tex_root_dir / 'gibbs_ces'


def make_idata_filename(model_name: str, period_label: str) -> str:
    parts = model_name.lower().split('_')
    infl = parts[0]
    family = parts[1]
    if parts[2] == 'orth':
        corr = 'uncorr'
        gap = '_'.join(parts[3:])
    else:
        corr = 'corr'
        gap = '_'.join(parts[2:])
    return f"{infl}_{family}_{gap}_{period_label}_{corr}"

gibbs_module = importlib.reload(gibbs_module)
run_ces = gibbs_module.run_ces
draws_to_idata = gibbs_module.draws_to_idata

dict_idata = {}
dict_draws = {}
print('=== Run Gibbs CES models ===')

for infl, spec in inflation_specs.items():
    pi = np.asarray(data[spec['pi']], dtype=float)
    pi_prev = np.asarray(data[spec['pi_prev']], dtype=float)
    pi_expect = np.asarray(data[spec['pi_expect']], dtype=float)

    print(f'--- Inflation: {infl} ---')

    for x_name, (x, x_prev) in x_datasets.items():
        run_name = f"{infl.upper()}_CES_{x_name}"
        orth_run_name = f"{infl.upper()}_CES_orth_{x_name}"
        print(f'Running Gibbs model: {run_name}')
        draws = run_ces(
            pi=pi,
            pi_prev=pi_prev,
            pi_expect=pi_expect,
            x=x,
            x_prev=x_prev,
            prior_specs=prior_specs,
            n_iter=n_iter,
            burn=burn,
            thin=thin,
            rng=np.random.default_rng(seed),
            orth=False,
        )
        dict_draws[run_name] = draws
        dict_idata[run_name] = draws_to_idata(draws)
        print(f'Running Gibbs model: {orth_run_name}')
        orth_draws = run_ces(
            pi=pi,
            pi_prev=pi_prev,
            pi_expect=pi_expect,
            x=x,
            x_prev=x_prev,
            prior_specs=prior_specs,
            n_iter=n_iter,
            burn=burn,
            thin=thin,
            rng=np.random.default_rng(seed),
            orth=True,
        )
        dict_draws[orth_run_name] = orth_draws
        dict_idata[orth_run_name] = draws_to_idata(orth_draws)

print('=== All Gibbs CES models finished ===')

base_models_to_show = [
    'CES_unemp_gap',
    'CES_output_gap_BN',
    'CES_markup_BN_inv',
    'CES_markup_inv',
]

period_label = '1982_2012'

for infl in inflation_specs.keys():
    models_to_show = [f"{infl.upper()}_{m}" for m in base_models_to_show]
    orth_models_to_show = [m.replace('_CES_', '_CES_orth_') for m in models_to_show]
    all_models_to_show = models_to_show + orth_models_to_show
    dict_items_fill = {k: dict_idata[k] for k in all_models_to_show if k in dict_idata}

    display_hmc_results(
        dict_items_fill,
        prior_specs,
        models_to_show=all_models_to_show,
        tex_dir=tex_dir / infl.lower(),
        params=('alpha', 'kappa', 'phi_1', 'rho', 'sigma_e', 'sigma_zeta'),
        title=f'Gibbs CES Results ({infl.upper()})',
    )

    fig_dir = base_fig_dir / 'gibbs_ces' / infl.lower()
    fig_dir.mkdir(parents=True, exist_ok=True)

    display_hmc_posterior_prior(
        dict_items_fill,
        prior_specs,
        models_to_show=all_models_to_show,
        fig_dir=fig_dir,
        params=('alpha', 'kappa', 'phi_1'),
        title=f'Prior vs Posterior ({infl.upper()})',
    )

    idata_dir = idat_dir / f'gibbs_ces_{infl.lower()}'
    dict_items_save = {make_idata_filename(k, period_label): v for k, v in dict_items_fill.items()}
    save_idata_map(dict_items_save, idata_dir)

=== Run Gibbs CES models ===
--- Inflation: ppi ---
Running Gibbs model: PPI_CES_unemp_gap
Running Gibbs model: PPI_CES_orth_unemp_gap
Running Gibbs model: PPI_CES_output_gap_BN
Running Gibbs model: PPI_CES_orth_output_gap_BN
Running Gibbs model: PPI_CES_markup_BN_inv
Running Gibbs model: PPI_CES_orth_markup_BN_inv
Running Gibbs model: PPI_CES_markup_inv
Running Gibbs model: PPI_CES_orth_markup_inv
--- Inflation: cpi ---
Running Gibbs model: CPI_CES_unemp_gap
Running Gibbs model: CPI_CES_orth_unemp_gap
Running Gibbs model: CPI_CES_output_gap_BN
Running Gibbs model: CPI_CES_orth_output_gap_BN
Running Gibbs model: CPI_CES_markup_BN_inv
Running Gibbs model: CPI_CES_orth_markup_BN_inv
Running Gibbs model: CPI_CES_markup_inv
Running Gibbs model: CPI_CES_orth_markup_inv
=== All Gibbs CES models finished ===
Gibbs CES Results (PPI)
                         model       param      mean    ci_2.5   ci_97.5
0            PPI_CES_unemp_gap       alpha  0.783717  0.681693  0.883247
1            PPI_

In [2]:
data = build_dataset(data_dir)
data = data.loc['1988-03-31':'2017-12-31'].copy()
data['DATE'] = pd.to_datetime(data.index)

x_datasets = {
    'unemp_gap': (
        np.asarray(data['unemp_gap'], dtype=float),
        np.asarray(data['unemp_gap_prev'], dtype=float),
    ),
    'output_gap_BN': (
        np.asarray(data['output_gap_BN'], dtype=float),
        np.asarray(data['output_gap_BN_prev'], dtype=float),
    ),
    'markup_BN_inv': (
        np.asarray(data['markup_BN_inv'], dtype=float),
        np.asarray(data['markup_BN_inv_prev'], dtype=float),
    ),
    'markup_inv': (
        np.asarray(data['markup_inv'], dtype=float),
        np.asarray(data['markup_inv_prev'], dtype=float),
    ),
}

gibbs_module = importlib.reload(gibbs_module)
run_ces = gibbs_module.run_ces
draws_to_idata = gibbs_module.draws_to_idata

dict_idata = {}
dict_draws = {}
print('=== Run Gibbs CES models ===')

for infl, spec in inflation_specs.items():
    pi = np.asarray(data[spec['pi']], dtype=float)
    pi_prev = np.asarray(data[spec['pi_prev']], dtype=float)
    pi_expect = np.asarray(data[spec['pi_expect']], dtype=float)

    print(f'--- Inflation: {infl} ---')

    for x_name, (x, x_prev) in x_datasets.items():
        run_name = f"{infl.upper()}_CES_{x_name}"
        orth_run_name = f"{infl.upper()}_CES_orth_{x_name}"
        print(f'Running Gibbs model: {run_name}')
        draws = run_ces(
            pi=pi,
            pi_prev=pi_prev,
            pi_expect=pi_expect,
            x=x,
            x_prev=x_prev,
            prior_specs=prior_specs,
            n_iter=n_iter,
            burn=burn,
            thin=thin,
            rng=np.random.default_rng(seed),
            orth=False,
        )
        dict_draws[run_name] = draws
        dict_idata[run_name] = draws_to_idata(draws)
        print(f'Running Gibbs model: {orth_run_name}')
        orth_draws = run_ces(
            pi=pi,
            pi_prev=pi_prev,
            pi_expect=pi_expect,
            x=x,
            x_prev=x_prev,
            prior_specs=prior_specs,
            n_iter=n_iter,
            burn=burn,
            thin=thin,
            rng=np.random.default_rng(seed),
            orth=True,
        )
        dict_draws[orth_run_name] = orth_draws
        dict_idata[orth_run_name] = draws_to_idata(orth_draws)

print('=== All Gibbs CES models finished ===')

base_models_to_show = [
    'CES_unemp_gap',
    'CES_output_gap_BN',
    'CES_markup_BN_inv',
    'CES_markup_inv',
]

period_label = '1988_2017'

for infl in inflation_specs.keys():
    models_to_show = [f"{infl.upper()}_{m}" for m in base_models_to_show]
    orth_models_to_show = [m.replace('_CES_', '_CES_orth_') for m in models_to_show]
    all_models_to_show = models_to_show + orth_models_to_show
    dict_items_fill = {k: dict_idata[k] for k in all_models_to_show if k in dict_idata}

    display_hmc_results(
        dict_items_fill,
        prior_specs,
        models_to_show=all_models_to_show,
        tex_dir=tex_dir / infl.lower(),
        params=('alpha', 'kappa', 'phi_1', 'rho', 'sigma_e', 'sigma_zeta'),
        title=f'Gibbs CES Results ({infl.upper()})',
    )

    fig_dir = base_fig_dir / 'gibbs_ces' / infl.lower()
    fig_dir.mkdir(parents=True, exist_ok=True)

    display_hmc_posterior_prior(
        dict_items_fill,
        prior_specs,
        models_to_show=all_models_to_show,
        fig_dir=fig_dir,
        params=('alpha', 'kappa', 'phi_1'),
        title=f'Prior vs Posterior ({infl.upper()})',
    )

    idata_dir = idat_dir / f'gibbs_ces_{infl.lower()}'
    dict_items_save = {make_idata_filename(k, period_label): v for k, v in dict_items_fill.items()}
    save_idata_map(dict_items_save, idata_dir)




=== Run Gibbs CES models ===
--- Inflation: ppi ---
Running Gibbs model: PPI_CES_unemp_gap
Running Gibbs model: PPI_CES_orth_unemp_gap
Running Gibbs model: PPI_CES_output_gap_BN
Running Gibbs model: PPI_CES_orth_output_gap_BN
Running Gibbs model: PPI_CES_markup_BN_inv
Running Gibbs model: PPI_CES_orth_markup_BN_inv
Running Gibbs model: PPI_CES_markup_inv
Running Gibbs model: PPI_CES_orth_markup_inv
--- Inflation: cpi ---
Running Gibbs model: CPI_CES_unemp_gap
Running Gibbs model: CPI_CES_orth_unemp_gap
Running Gibbs model: CPI_CES_output_gap_BN
Running Gibbs model: CPI_CES_orth_output_gap_BN
Running Gibbs model: CPI_CES_markup_BN_inv
Running Gibbs model: CPI_CES_orth_markup_BN_inv
Running Gibbs model: CPI_CES_markup_inv
Running Gibbs model: CPI_CES_orth_markup_inv
=== All Gibbs CES models finished ===
Gibbs CES Results (PPI)
                         model       param      mean    ci_2.5   ci_97.5
0            PPI_CES_unemp_gap       alpha  0.782321  0.679766  0.884336
1            PPI_